In [1]:
import pandas
import chromadb
import langchain
import sentence_transformers
import streamlit
print("All libraries working!")

All libraries working!


In [2]:
import pandas as pd 
from pathlib import Path

data_folder = Path("data")
csv_files = list(data_folder.glob("*.csv")) 
print(csv_files)                          

[WindowsPath('data/customers.csv'), WindowsPath('data/employees.csv'), WindowsPath('data/orders.csv'), WindowsPath('data/order_items.csv'), WindowsPath('data/products.csv'), WindowsPath('data/stores.csv')]


In [3]:
dataframes = {}

for file in csv_files:
    df = pd.read_csv(file)
    dataframes[file.stem] = df 
    print(f"{file.stem} : {df.shape[0] } rows, {df.shape[1]} columns ")


customers : 2000 rows, 11 columns 
employees : 250 rows, 10 columns 
orders : 10000 rows, 10 columns 
order_items : 30021 rows, 9 columns 
products : 500 rows, 8 columns 
stores : 25 rows, 8 columns 


In [4]:
for name, df in dataframes.items():
    print(f"\n===== {name} =====")
    print(df.columns.to_list())
    print(df.head(2))


===== customers =====
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'province', 'country', 'created_timestamp', 'updated_timestamp', 'is_active']
   customer_id first_name  last_name                email  \
0            1     Ronald       Long    shall@example.com   
1            2     Carlos  Zimmerman  cesar33@example.org   

                   phone          city    province country  \
0  +1-372-617-6276x55955    Torresland      Alaska  Canada   
1             4587577121  Lake Whitney  California  Canada   

     created_timestamp    updated_timestamp is_active  
0  2026-06-13 03:01:47  2026-06-21 10:15:55         Y  
1  2026-06-23 23:57:34  2026-06-26 18:24:36         Y  

===== employees =====
['employee_id', 'store_id', 'first_name', 'last_name', 'email', 'job_title', 'salary', 'created_timestamp', 'updated_timestamp', 'is_active']
   employee_id  store_id first_name last_name                      email  \
0            1         9     Robert      Diaz  ang

In [5]:
for name, df in dataframes.items():
    dupes = df.duplicated().sum()
    empty = df.isnull().sum().sum()
    print(f"{name}: {dupes} duplicate rows, {empty} empty cells")

customers: 0 duplicate rows, 0 empty cells
employees: 0 duplicate rows, 0 empty cells
orders: 0 duplicate rows, 0 empty cells
order_items: 0 duplicate rows, 0 empty cells
products: 0 duplicate rows, 0 empty cells
stores: 0 duplicate rows, 0 empty cells


In [6]:
skip_columns = ["created_timestamp", "updated_timestamp", "is_active"]

def row_to_text(table_name, row):
    parts = [f"table: {table_name}"]
    for column, value in row.items():
        if column in skip_columns:
            continue
        parts.append(f"{column}: {value}")
    return " | ".join(parts)

In [7]:
for name, df in dataframes.items():
    first_row = df.iloc[0]
    print(row_to_text(name, first_row))
    print()

table: customers | customer_id: 1 | first_name: Ronald | last_name: Long | email: shall@example.com | phone: +1-372-617-6276x55955 | city: Torresland | province: Alaska | country: Canada

table: employees | employee_id: 1 | store_id: 9 | first_name: Robert | last_name: Diaz | email: angelarussell@example.com | job_title: IT consultant | salary: 52919.01

table: orders | order_id: 1 | customer_id: 522 | store_id: 10 | order_timestamp: 2026-12-02 01:28:27 | payment_method: Cash | order_status: Pending | total_amount: 4202.65

table: order_items | order_item_id: 1 | order_id: 1 | product_id: 214 | quantity: 3 | unit_price: 9.23 | line_amount: 27.69

table: products | product_id: 1 | product_name: After Product | category: Home | brand: Samsung | price: 164.31

table: stores | store_id: 1 | store_name: Walmart Store 1 | city: South Nicolestad | province: Illinois | country: Canada



In [8]:
from langchain_core.documents import Document

documents = []

for name, df in dataframes.items():
    for index, row in df.iterrows():
        text = row_to_text(name, row)
        doc = Document(
            page_content=text,
            metadata={"table": name, "row": int(index)}
        )
        documents.append(doc)

print(f"Total documents: {len(documents)}")
print(documents[0])

Total documents: 42796
page_content='table: customers | customer_id: 1 | first_name: Ronald | last_name: Long | email: shall@example.com | phone: +1-372-617-6276x55955 | city: Torresland | province: Alaska | country: Canada' metadata={'table': 'customers', 'row': 0}


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
vector = embedding_model.embed_query("Customer from delhi bought a laptop")
print(f"Length: {len(vector)}")
print(vector[:10])

Length: 384
[-0.0575544610619545, 0.05379227176308632, 0.02044915035367012, -0.08638865500688553, -0.019732993096113205, -0.029004866257309914, 0.05996743217110634, 0.02286757156252861, 0.04514944553375244, -0.03384176269173622]


In [11]:
import numpy as np 
sentences = [
    "customer from Delhi bought a laptop",
    "a buyer in New Delhi purchased a computer",
    "the store sells fresh vegetables",
    "employee salary is 50000"
]


vectors = embedding_model.embed_documents(sentences) 

def similarity (a,b) :
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b) )

base = vectors[0] 

for sentences , vec in zip(sentences,vectors) :
    score = similarity(base,vec)
    print(f"{score:.2f}  ->  {sentences}") 

1.00  ->  customer from Delhi bought a laptop
0.80  ->  a buyer in New Delhi purchased a computer
0.24  ->  the store sells fresh vegetables
0.16  ->  employee salary is 50000


In [12]:
import time

sample_texts = [doc.page_content for doc in documents[:500]]

start = time.time()
sample_vectors = embedding_model.embed_documents(sample_texts)
end = time.time()

seconds = end - start
print(f"Embedded {len(sample_vectors)} rows in {seconds:.1f} seconds")

estimate = seconds / 500 * len(documents)
print(f"Estimated time for all {len(documents)} rows: {estimate / 60:.1f} minutes")

Embedded 500 rows in 6.7 seconds
Estimated time for all 42796 rows: 9.5 minutes


In [13]:
from collections import Counter

skip_tables = ["order_items"]

rag_documents = [doc for doc in documents if doc.metadata["table"] not in skip_tables]

print(f"Documents for ChromaDB: {len(rag_documents)}")
print(Counter(doc.metadata["table"] for doc in rag_documents))

Documents for ChromaDB: 12775
Counter({'orders': 10000, 'customers': 2000, 'products': 500, 'employees': 250, 'stores': 25})


In [14]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="csv_data",
    embedding_function=embedding_model,
    persist_directory="chroma_db"
)

existing = vector_store._collection.count()
print(f"Documents already in ChromaDB: {existing}")

Documents already in ChromaDB: 12775


In [15]:
batch_size = 500

if existing == 0:
    for i in range(0, len(rag_documents), batch_size):
        batch = rag_documents[i : i + batch_size]
        vector_store.add_documents(batch)
        done = min(i + batch_size, len(rag_documents))
        print(f"Stored {done} / {len(rag_documents)}")
    print("All documents stored!")
else:
    print("Data already stored, skipping.")

Data already stored, skipping.


In [16]:
loaded_store = Chroma(
    collection_name="csv_data",
    embedding_function=embedding_model,
    persist_directory="chroma_db"
)

print(f"Total records: {loaded_store._collection.count()}")

sample = loaded_store._collection.peek(limit=1)
print("Text:", sample["documents"][0])
print("Metadata:", sample["metadatas"][0])
print("Vector length:", len(sample["embeddings"][0]))

Total records: 12775
Text: table: customers | customer_id: 1 | first_name: Ronald | last_name: Long | email: shall@example.com | phone: +1-372-617-6276x55955 | city: Torresland | province: Alaska | country: Canada
Metadata: {'table': 'customers', 'row': 0}
Vector length: 384


In [17]:
query = "employees working as consultants"

results = vector_store.similarity_search(query, k=5)

for doc in results:
    print(doc.metadata["table"], "->", doc.page_content)
    print()

employees -> table: employees | employee_id: 1 | store_id: 9 | first_name: Robert | last_name: Diaz | email: angelarussell@example.com | job_title: IT consultant | salary: 52919.01

employees -> table: employees | employee_id: 94 | store_id: 20 | first_name: Christian | last_name: Smith | email: stacey99@example.com | job_title: Management consultant | salary: 60758.49

employees -> table: employees | employee_id: 217 | store_id: 3 | first_name: Charles | last_name: Torres | email: michaelschultz@example.net | job_title: Management consultant | salary: 64593.82

employees -> table: employees | employee_id: 167 | store_id: 13 | first_name: Stacy | last_name: Arellano | email: davisjeremy@example.net | job_title: Doctor, general practice | salary: 36402.42

employees -> table: employees | employee_id: 250 | store_id: 19 | first_name: Gary | last_name: Haynes | email: christophergriffin@example.org | job_title: Speech and language therapist | salary: 52216.38



In [18]:
query = "employees working as consultants"

results = vector_store.similarity_search_with_score(query, k=5)

for doc, score in results:
    print(f"{score:.3f} | {doc.page_content}")
    print()

1.155 | table: employees | employee_id: 1 | store_id: 9 | first_name: Robert | last_name: Diaz | email: angelarussell@example.com | job_title: IT consultant | salary: 52919.01

1.161 | table: employees | employee_id: 94 | store_id: 20 | first_name: Christian | last_name: Smith | email: stacey99@example.com | job_title: Management consultant | salary: 60758.49

1.196 | table: employees | employee_id: 217 | store_id: 3 | first_name: Charles | last_name: Torres | email: michaelschultz@example.net | job_title: Management consultant | salary: 64593.82

1.256 | table: employees | employee_id: 167 | store_id: 13 | first_name: Stacy | last_name: Arellano | email: davisjeremy@example.net | job_title: Doctor, general practice | salary: 36402.42

1.267 | table: employees | employee_id: 250 | store_id: 19 | first_name: Gary | last_name: Haynes | email: christophergriffin@example.org | job_title: Speech and language therapist | salary: 52216.38



In [19]:
query = "Samsung products in the home category"

results = vector_store.similarity_search_with_score(
    query,
    k=5,
    filter={"table": "products"}
)

for doc, score in results:
    print(f"{score:.3f} | {doc.page_content}")
    print()

0.833 | table: products | product_id: 428 | product_name: Throw Product | category: Home | brand: Samsung | price: 409.61

0.852 | table: products | product_id: 138 | product_name: Protect Product | category: Home | brand: Samsung | price: 92.73

0.863 | table: products | product_id: 319 | product_name: Sing Product | category: Home | brand: Samsung | price: 206.63

0.863 | table: products | product_id: 76 | product_name: Just Product | category: Home | brand: Samsung | price: 479.26

0.870 | table: products | product_id: 297 | product_name: Soon Product | category: Home | brand: Samsung | price: 125.03



In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

print("Key loaded:", groq_key is not None)
print("Starts with:", groq_key[:4] if groq_key else "nothing")

Key loaded: True
Starts with: gsk_


In [21]:
from groq import Groq

client = Groq(api_key=groq_key)
models = client.models.list()

for m in models.data:
    print(m.id)

qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
allam-2-7b
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m


In [22]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=groq_key
)

response = llm.invoke("Explain what a vector database is in 2 simple sentences.")
print(response.content)

A vector database stores data as high‑dimensional numeric vectors (often embeddings that capture the meaning of text, images, etc.) and indexes them for fast similarity search. This lets you quickly find items that are “close” to a query vector, enabling applications like semantic search, recommendation, and nearest‑neighbor retrieval.


In [23]:
response = llm.invoke("Which Samsung products are in the Home category in my store?")
print(response.content)


I’m not able to see the inventory details for your specific store, so I can’t list the Samsung items you currently have in the **Home** category. 

If you have access to your store’s product database or inventory management system, you can usually find that information by:

1. **Filtering by Brand** – Look for a “Brand” or “Manufacturer” field and select **Samsung**.  
2. **Filtering by Category** – Then apply a second filter for the **Home** (or “Home & Kitchen”) category.  
3. **Running a Report** – Many systems let you export the filtered list to a spreadsheet or PDF for easy review.

If you let me know which platform or software you use to manage your inventory (e.g., Shopify, Lightspeed, a custom ERP, etc.), I can give you more detailed steps on how to run that query.


In [24]:
question = "Which Samsung products are in the Home category?"

docs = vector_store.similarity_search(question, k=5, filter={"table": "products"})

context = "\n".join(doc.page_content for doc in docs)

prompt = f"""You are a helpful data assistant.
Answer the question using ONLY the data below.
If the answer is not in the data, say "I don't know based on the data."

Data:
{context}

Question: {question}
Answer:"""

print(prompt)

You are a helpful data assistant.
Answer the question using ONLY the data below.
If the answer is not in the data, say "I don't know based on the data."

Data:
table: products | product_id: 297 | product_name: Soon Product | category: Home | brand: Samsung | price: 125.03
table: products | product_id: 316 | product_name: Suffer Product | category: Electronics | brand: Samsung | price: 243.99
table: products | product_id: 428 | product_name: Throw Product | category: Home | brand: Samsung | price: 409.61
table: products | product_id: 319 | product_name: Sing Product | category: Home | brand: Samsung | price: 206.63
table: products | product_id: 178 | product_name: Behavior Product | category: Electronics | brand: Samsung | price: 434.53

Question: Which Samsung products are in the Home category?
Answer:


In [25]:
response = llm.invoke(prompt)
print(response.content)

The Samsung products in the **Home** category are:

- Soon Product (product_id 297)  
- Throw Product (product_id 428)  
- Sing Product (product_id 319)


In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful data assistant. "
     "Answer using ONLY the data provided. "
     "If the answer is not in the data, say \"I don't know based on the data.\" "
     "Mention IDs and names when relevant."),
    ("human", "Data:\n{context}\n\nQuestion: {question}")
])

def ask(question, table=None, k=10):
    search_filter = {"table": table} if table else None
    docs = vector_store.similarity_search(question, k=k, filter=search_filter)
    context = "\n".join(doc.page_content for doc in docs)
    messages = prompt_template.invoke({"context": context, "question": question})
    response = llm.invoke(messages)
    return response.content, docs

In [27]:
answer, sources = ask("Which Samsung products are in the Home category?", table="products")

print(answer)
print(f"\nRows given to the LLM: {len(sources)}")

The Samsung products that belong to the **Home** category are:

- **product_id 297** – Soon Product  
- **product_id 428** – Throw Product  
- **product_id 319** – Sing Product  
- **product_id 138** – Protect Product  
- **product_id 118** – State Product  
- **product_id 58** – Start Product  
- **product_id 366** – Describe Product

Rows given to the LLM: 10


In [28]:
products = dataframes["products"]

truth = products[(products["brand"] == "Samsung") & (products["category"] == "Home")]

print(f"Actual Samsung Home products: {len(truth)}")
print(truth[["product_id", "product_name"]])

Actual Samsung Home products: 16
     product_id        product_name
0             1       After Product
57           58       Start Product
65           66      Manage Product
71           72        Hour Product
75           76        Just Product
117         118       State Product
137         138     Protect Product
163         164         Day Product
199         200  Difference Product
284         285        Also Product
296         297        Soon Product
318         319        Sing Product
365         366    Describe Product
409         410       Above Product
427         428       Throw Product
461         462  Experience Product


In [29]:
test_questions = [
    ("Tell me about customer Ronald Long", "customers"),
    ("What is the job title and salary of Charles Torres?", "employees"),
    ("Which store is located in Kristaberg?", "stores"),
    ("How many orders were cancelled?", "orders"),
    ("What is the most expensive product?", "products"),
]

for question, table in test_questions:
    answer, sources = ask(question, table=table)
    print("Q:", question)
    print("A:", answer)
    print("-" * 60)

Q: Tell me about customer Ronald Long
A: I don't know based on the data.
------------------------------------------------------------
Q: What is the job title and salary of Charles Torres?
A: Charles Torres (employee_id 217) holds the job title **Management consultant** and has a salary of **64,593.82**.
------------------------------------------------------------
Q: Which store is located in Kristaberg?
A: The store located in **Kristaberg** is **Walmart Store 22** (store_id 22).
------------------------------------------------------------
Q: How many orders were cancelled?
A: Based on the provided data, there are **10** orders with an `order_status` of **Cancelled**.
------------------------------------------------------------
Q: What is the most expensive product?
A: The most expensive product is **product_id 260 – “Mrs Product”**, priced at **$370.89**.
------------------------------------------------------------


In [30]:
print(dataframes["orders"]["order_status"].value_counts())
print(dataframes["products"].sort_values("price", ascending=False).head(1))

order_status
Cancelled    2587
Returned     2530
Completed    2447
Pending      2436
Name: count, dtype: int64
    product_id   product_name category    brand   price    created_timestamp  \
57          58  Start Product     Home  Samsung  498.66  2026-05-05 07:37:31   

      updated_timestamp is_active  
57  2026-05-13 02:05:13         Y  


In [31]:
import sqlite3

conn = sqlite3.connect(":memory:", check_same_thread=False)

for name, df in dataframes.items():
    df.to_sql(name, conn, index=False, if_exists="replace")

print("Tables loaded:", list(dataframes.keys()))

Tables loaded: ['customers', 'employees', 'orders', 'order_items', 'products', 'stores']


In [32]:
query = """
SELECT order_status, COUNT(*) AS total
FROM orders
GROUP BY order_status
"""

display(pd.read_sql(query, conn))

,order_status,total
0,Cancelled,2587
1,Completed,2447
2,Pending,2436
3,Returned,2530


In [33]:
query = """
SELECT product_name, price from products
order by price DESC 
limit 1
"""

display(pd.read_sql(query, conn))

,product_name,price
0,Start Product,498.66


In [34]:
def get_schema(conn):
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
    schema = ""
    for table in tables:
        schema += f"\nTable: {table}\n"
        cols = pd.read_sql(f"SELECT name, type FROM pragma_table_info('{table}')", conn)
        for _, col in cols.iterrows():
            line = f"  - {col['name']} ({col['type']})"
            if col["type"] == "TEXT":
                values = pd.read_sql(
                    f'SELECT DISTINCT "{col["name"]}" FROM {table} LIMIT 11', conn
                ).iloc[:, 0].tolist()
                if len(values) <= 10:
                    line += f" values: {values}"
            schema += line + "\n"
    return schema

schema = get_schema(conn)
print(schema)


Table: customers
  - customer_id (INTEGER)
  - first_name (TEXT)
  - last_name (TEXT)
  - email (TEXT)
  - phone (TEXT)
  - city (TEXT)
  - province (TEXT)
  - country (TEXT) values: ['Canada']
  - created_timestamp (TEXT)
  - updated_timestamp (TEXT)
  - is_active (TEXT) values: ['Y', 'N']

Table: employees
  - employee_id (INTEGER)
  - store_id (INTEGER)
  - first_name (TEXT)
  - last_name (TEXT)
  - email (TEXT)
  - job_title (TEXT)
  - salary (REAL)
  - created_timestamp (TEXT)
  - updated_timestamp (TEXT)
  - is_active (TEXT) values: ['Y', 'N']

Table: orders
  - order_id (INTEGER)
  - customer_id (INTEGER)
  - store_id (INTEGER)
  - order_timestamp (TEXT)
  - payment_method (TEXT) values: ['Cash', 'Debit Card', 'Credit Card', 'Gift Card', 'Online']
  - order_status (TEXT) values: ['Pending', 'Cancelled', 'Returned', 'Completed']
  - total_amount (REAL)
  - created_timestamp (TEXT)
  - updated_timestamp (TEXT)
  - is_active (TEXT) values: ['Y', 'N']

Table: order_items
  - order_

In [35]:
sql_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at writing SQLite queries.\n"
     "Given the database schema below, write ONE SQLite query that answers the user's question.\n"
     "Rules:\n"
     "- Use only the tables and columns in the schema.\n"
     "- Use SQLite syntax.\n"
     "- Only write SELECT queries. Never write INSERT, UPDATE, DELETE, DROP or ALTER.\n"
     "- Do not add extra filters (such as is_active) unless the user asks for them.\n"
     "- Return ONLY the SQL query. No explanation, no markdown.\n\n"
     "Schema:\n{schema}"),
    ("human", "{question}")
])

def write_sql(question):
    messages = sql_prompt.invoke({"schema": schema, "question": question})
    response = llm.invoke(messages)
    sql = response.content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql

In [36]:
test_questions = [
    "How many orders were cancelled?",
    "What is the most expensive product?",
    "What products did customer Ronald Long order?",
]

for q in test_questions:
    print("Q:", q)
    print(write_sql(q))
    print("-" * 60)

Q: How many orders were cancelled?
SELECT COUNT(*) FROM orders WHERE order_status = 'Cancelled';
------------------------------------------------------------
Q: What is the most expensive product?
SELECT product_id, product_name, price
FROM products
ORDER BY price DESC
LIMIT 1
------------------------------------------------------------
Q: What products did customer Ronald Long order?
SELECT DISTINCT p.product_name
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE c.first_name = 'Ronald' AND c.last_name = 'Long';
------------------------------------------------------------


In [37]:
import re

def is_safe(sql):
    cleaned = sql.strip().lower()
    if not (cleaned.startswith("select") or cleaned.startswith("with")):
        return False
    blocked = ["insert", "update", "delete", "drop", "alter", "create", "attach", "pragma"]
    words = re.findall(r"[a-z_]+", cleaned)
    return not any(word in blocked for word in words)


def run_sql(question, max_tries=2):
    sql = write_sql(question)
    error = None
    for attempt in range(max_tries):
        if not is_safe(sql):
            return sql, None, "Blocked: only read-only SELECT queries are allowed."
        try:
            result = pd.read_sql(sql, conn)
            return sql, result, None
        except Exception as e:
            error = str(e)
            retry_question = (
                f"{question}\n\nYour previous query was:\n{sql}\n"
                f"It failed with this error: {error}\nWrite a corrected query."
            )
            sql = write_sql(retry_question)
    return sql, None, error

In [38]:
sql, result, error = run_sql("What products did customer Ronald Long order?")
print(sql)
print("Error:", error)
display(result)

print(is_safe("DELETE FROM orders"))
print(is_safe("SELECT updated_timestamp FROM orders"))

SELECT DISTINCT p.product_id, p.product_name, p.category, p.brand, p.price
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE c.first_name = 'Ronald' AND c.last_name = 'Long';
Error: None


,product_id,product_name,category,brand,price
0,74,Although Product,Grocery,Sony,97.10
1,176,Behavior Product,Sports,Adidas,455.35
2,333,Throw Product,Clothing,Nestle,244.29
3,130,Style Product,Clothing,Adidas,401.41
4,369,Pretty Product,Toys,Apple,114.26
5,392,Throw Product,Clothing,Nestle,148.48
6,164,Day Product,Home,Samsung,38.12
7,192,Glass Product,Toys,Nestle,111.10
8,210,Outside Product,Clothing,LG,25.78
9,479,Moment Product,Sports,Nike,193.32


False
True


In [39]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful data assistant. "
     "Answer the user's question in clear, simple English using ONLY the SQL result provided. "
     "If the result is empty, say that no matching data was found."),
    ("human", "Question: {question}\n\nSQL used:\n{sql}\n\nResult:\n{result}")
])

def ask_sql(question):
    sql, result, error = run_sql(question)
    if error:
        return f"Sorry, I couldn't answer that. ({error})", sql, None
    result_text = result.head(50).to_string(index=False)
    if len(result) > 50:
        result_text += f"\n... and {len(result) - 50} more rows"
    messages = answer_prompt.invoke({"question": question, "sql": sql, "result": result_text})
    response = llm.invoke(messages)
    return response.content, sql, result

In [40]:
test_questions = [
    "How many orders were cancelled?",
    "What is the most expensive product?",
    "Tell me about customer Ronald Long",
    "List all Samsung products in the Home category",
]

for q in test_questions:
    answer, sql, result = ask_sql(q)
    print("Q:", q)
    print("A:", answer)
    print("-" * 60)

Q: How many orders were cancelled?
A: 2587 orders were cancelled.
------------------------------------------------------------
Q: What is the most expensive product?
A: The most expensive product is **Start Product** (product ID 58) with a price of **$498.66**.
------------------------------------------------------------
Q: Tell me about customer Ronald Long
A: Ronald Long is a customer with the following details:

- **Customer ID:** 1  
- **Email:** shall@example.com  
- **Phone:** +1-372-617-6276x55955  
- **City:** Torresland  
- **Province/State:** Alaska  
- **Country:** Canada  
- **Account created on:** 2026‑06‑13 at 03:01:47  
- **Last updated on:** 2026‑06‑21 at 10:15:55  
- **Account status:** Active (Y)
------------------------------------------------------------
Q: List all Samsung products in the Home category
A: Here are the Samsung products that belong to the **Home** category:

| product_id | product_name          |
|------------|----------------------|
| 1          | A

In [41]:
import json

rag_tables = [t for t in dataframes.keys() if t not in skip_tables]

router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a router for a data chatbot. Decide how to answer the user's question.\n"
     "Choose 'sql' if the question needs exact values, counting, totals, averages, max or min, "
     "rankings, listing ALL matching rows, specific names or IDs, or combining tables.\n"
     "Choose 'rag' if the question is fuzzy or descriptive, about meaning or similarity, "
     "such as recommendations or 'find something like X'.\n"
     "If you choose 'rag', also pick the single most relevant table from this list: {tables}\n"
     "Respond ONLY with JSON, for example: {{\"route\": \"rag\", \"table\": \"products\"}}"),
    ("human", "{question}")
])

def route_question(question):
    messages = router_prompt.invoke({"tables": ", ".join(rag_tables), "question": question})
    response = llm.invoke(messages)
    text = response.content.strip().replace("```json", "").replace("```", "").strip()
    try:
        decision = json.loads(text)
    except json.JSONDecodeError:
        decision = {"route": "sql", "table": None}
    if decision.get("route") not in ("sql", "rag"):
        decision["route"] = "sql"
    if decision.get("table") not in rag_tables:
        decision["table"] = None
    return decision

In [42]:
test_questions = [
    "How many customers are from Alaska?",
    "Suggest some products for someone who loves sports",
    "What is the total revenue per store?",
    "Find employees who work in healthcare-type jobs",
    "Which customer placed the most orders?",
]

for q in test_questions:
    print(q, "->", route_question(q))

How many customers are from Alaska? -> {'route': 'sql', 'table': None}
Suggest some products for someone who loves sports -> {'route': 'rag', 'table': 'products'}
What is the total revenue per store? -> {'route': 'sql', 'table': None}
Find employees who work in healthcare-type jobs -> {'route': 'sql', 'table': None}
Which customer placed the most orders? -> {'route': 'sql', 'table': None}


In [43]:
def chat(question):
    decision = route_question(question)

    if decision["route"] == "sql":
        answer, sql, result = ask_sql(question)
        return {"answer": answer, "route": "sql", "sql": sql, "table": None}
    else:
        answer, sources = ask(question, table=decision["table"])
        return {"answer": answer, "route": "rag", "sql": None, "table": decision["table"]}

In [44]:
final_test = [
    "Tell me about customer Ronald Long",
    "What is the job title and salary of Charles Torres?",
    "Which store is located in Kristaberg?",
    "How many orders were cancelled?",
    "What is the most expensive product?",
    "Suggest some products for someone who loves sports",
    "Find employees who work in healthcare-type jobs",
]

for q in final_test:
    r = chat(q)
    print("Q:", q)
    print(f"Route: {r['route']}  Table: {r['table']}")
    print("A:", r["answer"])
    print("-" * 60)

Q: Tell me about customer Ronald Long
Route: sql  Table: None
A: Ronald Long is a customer with the following details:

- **Customer ID:** 1  
- **Email:** shall@example.com  
- **Phone:** +1-372-617-6276x55955  
- **City:** Torresland  
- **Province/State:** Alaska  
- **Country:** Canada  
- **Account created on:** 2026‑06‑13 at 03:01:47  
- **Last updated on:** 2026‑06‑21 at 10:15:55  
- **Account status:** Active (Y)
------------------------------------------------------------
Q: What is the job title and salary of Charles Torres?
Route: sql  Table: None
A: The job title is **Management consultant** and the salary is **$64,593.82**.
------------------------------------------------------------
Q: Which store is located in Kristaberg?
Route: sql  Table: None
A: Walmart Store 22.
------------------------------------------------------------
Q: How many orders were cancelled?
Route: sql  Table: None
A: 2587 orders were cancelled.
---------------------------------------------------------

In [45]:
print(chat("Find employees who work in healthcare-type jobs")["sql"])

SELECT employee_id, first_name, last_name, job_title
FROM employees
WHERE job_title LIKE '%health%'
   OR job_title LIKE '%nurse%'
   OR job_title LIKE '%doctor%'
   OR job_title LIKE '%pharma%'
   OR job_title LIKE '%clinic%'
   OR job_title LIKE '%medical%';


In [46]:
import importlib
import engine

importlib.reload(engine)
print("engine.py loaded OK")

engine.py loaded OK


In [47]:
importlib.reload(engine)

test_engine = engine.build_engine(
    dataframes,
    embedding_model,
    skip_tables=["order_items", "orders"]
)
test_llm = engine.load_llm()

for q in [
    "How many orders were cancelled?",
    "Which store is located in Kristaberg?",
    "Suggest some products for someone who loves sports",
]:
    r = engine.chat(test_engine, test_llm, q)
    print("Q:", q)
    print(f"Route: {r['route']}  Table: {r['table']}")
    print("A:", r["answer"])
    print("-" * 60)

Q: How many orders were cancelled?
Route: sql  Table: None
A: 2587 orders were cancelled.
------------------------------------------------------------
Q: Which store is located in Kristaberg?
Route: sql  Table: None
A: Walmart Store 22.
------------------------------------------------------------
Q: Suggest some products for someone who loves sports
Route: rag  Table: products
A: Here are a few sports‑category products you might like:

- **Product ID 421 – “Better Product”** (Brand: Nike, Price: $419.84)  
- **Product ID 115 – “Kid Product”** (Brand: Apple, Price: $473.68)  
- **Product ID 429 – “Nature Product”** (Brand: Adidas, Price: $282.88)  
- **Product ID 217 – “Particularly Product”** (Brand: Nike, Price: $108.18)  
- **Product ID 304 – “Laugh Product”** (Brand: Adidas, Price: $496.77)  
------------------------------------------------------------


In [49]:
importlib.reload(engine)

for q in [
    "Show all columns of the products table",
    "What is the total revenue per store?",
    "How many orders were cancelled?",
]:
    r = engine.chat(test_engine, test_llm, q)
    print("Q:", q)
    print("Route:", r["route"])
    print("A:", r["answer"])
    if r["result"] is not None:
        print("Result table size:", r["result"].shape)
    print("-" * 60)
    

Q: Show all columns of the products table
Route: schema
A: **Products table – all columns**

- product_id  
- product_name  
- category  
- brand  
- price  
- created_timestamp  
- updated_timestamp  
- is_active  
------------------------------------------------------------
Q: What is the total revenue per store?
Route: sql
A: The query returned revenue totals for all 25 Walmart stores. Store 10 leads with about **$840,945.79** in sales, followed closely by Store 23 at **$834,146.53** and Store 22 at **$827,663.99**. The mid‑range stores (e.g., Store 12, Store 4, Store 19) each generated roughly **$750,000–$776,000**. At the lower end, Store 14 recorded the smallest total revenue of about **$659,439.39**.
Result table size: (25, 3)
------------------------------------------------------------
Q: How many orders were cancelled?
Route: sql
A: There were **2,587** orders with a status of “Cancelled.”
Result table size: (1, 1)
------------------------------------------------------------
